# Chunk (5 Strategies) -> Build Vector Index

**Objective**: Transform crawled docs into retrieval chunks using five strategies, embed with OpenRouter/OpenAI, build ChromaDB persistent index.

**Architecture**: Uses chunking functions from `context_engineering.application.ingest_documents_service.chunkers`

**Provider Support**: Uses OpenRouter unified API (access OpenAI, Anthropic, Google, etc. with one key) or direct OpenAI

In [2]:
# Cell 2: Imports & Environment Setup
import os
import sys
import json
import random
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

# Load configuration
from config import (
    CRAWL_OUT_DIR, VECTOR_DIR, EMBEDDING_MODEL, PROVIDER
)

random.seed(42)

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: OpenAI
📁 Project root: e:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands


## Import Chunking Services

Using chunking functions from application layer (NOT defined here!)

In [3]:
# Cell 3: Import Chunking Services
from application.ingest_document_service import (
    semantic_chunk,
    fixed_chunk,
    sliding_chunk,
    parent_child_chunk,
    late_chunk_index,
    late_chunk_split,
    count_tokens
)

print("✅ Chunking services loaded from service layer")
print("📍 Location: application.ingest_document_service.chunkers")
print("\n🎯 Available strategies:")
print("   1. semantic_chunk  - Split by heading structure")
print("   2. fixed_chunk     - Uniform 800-token chunks with overlap")
print("   3. sliding_chunk   - Overlapping windows for better recall")
print("   4. parent_child_chunk - Child chunks plus parent context")
print("   5. late_chunk_index   - Large base chunks for retrieval-time splitting")

✅ Chunking services loaded from service layer
📍 Location: application.ingest_document_service.chunkers

🎯 Available strategies:
   1. semantic_chunk  - Split by heading structure
   2. fixed_chunk     - Uniform 800-token chunks with overlap
   3. sliding_chunk   - Overlapping windows for better recall
   4. parent_child_chunk - Child chunks plus parent context
   5. late_chunk_index   - Large base chunks for retrieval-time splitting


## Load Corpus

In [ ]:
# Cell 4: Load Corpus
jsonl_path = CRAWL_OUT_DIR / "primelands_docs.jsonl"

if not jsonl_path.exists():
    raise FileNotFoundError(f"❌ Corpus not found. Run 01_crawl_primelands.ipynb first.")

with open(jsonl_path, 'r', encoding='utf-8') as f:
    documents = [json.loads(line) for line in f]

print(f"✅ Loaded {len(documents)} documents")
print(f"📊 Total content size: {sum(len(d['headings']) for d in documents):,} chars")

✅ Loaded 140 documents
📊 Total content size: 8,161 chars


## Apply Chunking Strategies

In [5]:
# Cell: Cleanup Vector Store (prevents corruption)
import shutil

# Remove existing vector store to prevent corruption issues
if VECTOR_DIR.exists():
    print(f"🗑️  Removing existing vector store: {VECTOR_DIR}")
    shutil.rmtree(VECTOR_DIR)
    print("   ✅ Cleaned up")

# Create fresh directory
VECTOR_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Fresh vector directory ready: {VECTOR_DIR}")


🗑️  Removing existing vector store: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\vectorstore
   ✅ Cleaned up
📁 Fresh vector directory ready: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\vectorstore


In [6]:
# Cell 5: Semantic Chunking (using service!)
print("🔄 Running semantic chunking...")
semantic_chunks = semantic_chunk(documents)

# Save
semantic_path = CRAWL_OUT_DIR / "chunks_semantic.jsonl"
with open(semantic_path, 'w', encoding='utf-8') as f:
    for chunk in semantic_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Semantic chunking complete: {len(semantic_chunks)} chunks")
print(f"💾 Saved to: {semantic_path}")

🔄 Running semantic chunking...
✅ Semantic chunking complete: 129 chunks
💾 Saved to: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\chunks_semantic.jsonl


In [7]:
# Cell 6: Fixed-Window Chunking (using service!)
print("🔄 Running fixed-window chunking...")
fixed_chunks = fixed_chunk(documents)

# Save
fixed_path = CRAWL_OUT_DIR / "chunks_fixed.jsonl"
with open(fixed_path, 'w', encoding='utf-8') as f:
    for chunk in fixed_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

avg_tokens = sum(c['token_count'] for c in fixed_chunks) / len(fixed_chunks) if fixed_chunks else 0
print(f"✅ Fixed chunking complete: {len(fixed_chunks)} chunks")
print(f"📊 Avg token count: {avg_tokens:.1f}")
print(f"💾 Saved to: {fixed_path}")

🔄 Running fixed-window chunking...
✅ Fixed chunking complete: 140 chunks
📊 Avg token count: 205.3
💾 Saved to: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\chunks_fixed.jsonl


In [8]:
# Cell 7: Sliding-Window Chunking (using service!)
print("🔄 Running sliding-window chunking...")
sliding_chunks = sliding_chunk(documents)

# Save
sliding_path = CRAWL_OUT_DIR / "chunks_sliding.jsonl"
with open(sliding_path, 'w', encoding='utf-8') as f:
    for chunk in sliding_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Sliding chunking complete: {len(sliding_chunks)} chunks")
print(f"💾 Saved to: {sliding_path}")

🔄 Running sliding-window chunking...
✅ Sliding chunking complete: 141 chunks
💾 Saved to: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\chunks_sliding.jsonl


In [9]:
# Cell 8: Parent-Child Chunking (using service!)
print("🔄 Running parent-child chunking...")
child_chunks, parent_chunks = parent_child_chunk(documents)

# Save children and parents separately
child_path = CRAWL_OUT_DIR / "chunks_parent_child_children.jsonl"
with open(child_path, 'w', encoding='utf-8') as f:
    for chunk in child_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

parent_path = CRAWL_OUT_DIR / "chunks_parent_child_parents.jsonl"
with open(parent_path, 'w', encoding='utf-8') as f:
    for chunk in parent_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Parent-child chunking complete: {len(child_chunks)} children, {len(parent_chunks)} parents")
print(f"💾 Children saved to: {child_path}")
print(f"💾 Parents saved to: {parent_path}")

🔄 Running parent-child chunking...
✅ Parent-child chunking complete: 141 children, 140 parents
💾 Children saved to: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\chunks_parent_child_children.jsonl
💾 Parents saved to: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\chunks_parent_child_parents.jsonl


In [10]:
# Cell 9: Late Chunking (using service!)
print("🔄 Running late chunk indexing...")
late_chunks = late_chunk_index(documents)

# Save base passages for retrieval-time splitting
late_path = CRAWL_OUT_DIR / "chunks_late.jsonl"
with open(late_path, 'w', encoding='utf-8') as f:
    for chunk in late_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Late chunk indexing complete: {len(late_chunks)} base passages")
print(f"💾 Saved to: {late_path}")

🔄 Running late chunk indexing...
✅ Late chunk indexing complete: 140 base passages
💾 Saved to: E:\Tutorials\Zuu\AI Engineer Essentials\Mini Projects\Mini Project 2\Real Estate Intelligence Platform for Prime Lands\data\chunks_late.jsonl


## Spot-Check Samples

In [11]:
# Cell 10: Spot-Check Samples
print("🔍 Spot-Check: 2 samples from each chunk output\n")

def print_sample(chunk, strategy_name):
    print(f"**{strategy_name}** chunk:")
    print(f"  URL: {chunk['url']}")
    print(f"  Strategy: {chunk['strategy']}")
    print(f"  Text length: {len(chunk['text'])} chars")
    print(f"  Preview: {chunk['text'][:100]}...")
    print()

print("=" * 60)
print("SEMANTIC SAMPLES")
print("=" * 60)
for chunk in random.sample(semantic_chunks, min(2, len(semantic_chunks))):
    print_sample(chunk, "Semantic")

print("=" * 60)
print("FIXED-WINDOW SAMPLES")
print("=" * 60)
for chunk in random.sample(fixed_chunks, min(2, len(fixed_chunks))):
    print_sample(chunk, "Fixed")

print("=" * 60)
print("SLIDING-WINDOW SAMPLES")
print("=" * 60)
for chunk in random.sample(sliding_chunks, min(2, len(sliding_chunks))):
    print_sample(chunk, "Sliding")

print("=" * 60)
print("PARENT-CHILD CHILD SAMPLES")
print("=" * 60)
for chunk in random.sample(child_chunks, min(2, len(child_chunks))):
    print_sample(chunk, "Parent-Child Child")

print("=" * 60)
print("PARENT CONTEXT SAMPLES")
print("=" * 60)
for chunk in random.sample(parent_chunks, min(2, len(parent_chunks))):
    print_sample(chunk, "Parent")

print("=" * 60)
print("LATE-CHUNK BASE SAMPLES")
print("=" * 60)
for chunk in random.sample(late_chunks, min(2, len(late_chunks))):
    print_sample(chunk, "Late Chunk Base")

🔍 Spot-Check: 2 samples from each chunk output

SEMANTIC SAMPLES
**Semantic** chunk:
  URL: https://www.primelands.lk/land/DAGNY-GODAGAMA/en
  Strategy: semantic
  Text length: 600 chars
  Preview: [![primelogo.png](https://www.primelands.lk/public/assets/images/primelogo.png)](https://www.primela...

**Semantic** chunk:
  URL: https://www.primelands.lk/portfolio-property/MATARA-WELIGAMA/en
  Strategy: semantic
  Text length: 630 chars
  Preview: [![primelogo.png](https://www.primelands.lk/public/assets/images/primelogo.png)](https://www.primela...

FIXED-WINDOW SAMPLES
**Fixed** chunk:
  URL: https://www.primelands.lk/land/district/Kegalle/en
  Strategy: fixed
  Text length: 598 chars
  Preview: [![primelogo.png](https://www.primelands.lk/public/assets/images/primelogo.png)](https://www.primela...

**Fixed** chunk:
  URL: https://www.primelands.lk/land/city/Nuwara-Eliya/en
  Strategy: fixed
  Text length: 600 chars
  Preview: [![primelogo.png](https://www.primelands.lk/public/assets/i

## Build ChromaDB Vector Index

In [12]:
# Cell 11: Build Vector Index with LangChain
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from infra.llm_providers import get_default_embeddings

# Initialize embeddings using service factory (supports OpenRouter)
embeddings = get_default_embeddings()

print(f"✅ Embeddings initialized: {EMBEDDING_MODEL}")
print(f"🌐 Provider: {PROVIDER}")

# Prepare directory
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

# Combine all chunks
all_chunks = semantic_chunks + fixed_chunks + sliding_chunks + child_chunks + late_chunks
print(f"📊 Total chunks to embed: {len(all_chunks)}")

# Convert to LangChain Documents
lc_documents = []
for chunk in all_chunks:
    doc = Document(
        page_content=chunk['text'],
        metadata={
            "url": chunk['url'],
            "title": chunk['title'],
            "strategy": chunk['strategy'],
            "chunk_index": chunk['chunk_index']
        }
    )
    lc_documents.append(doc)

print(f"\n🚀 Creating Qdrant vector store...\n")

# Create vector store (LangChain handles batching + retries)
vectorstore = QdrantVectorStore.from_documents(
    documents=lc_documents,
    embedding=embeddings,
    path=str(VECTOR_DIR),
    collection_name="primelands"
)

print(f"✅ Vector store created!")

client = vectorstore.client
count_result = client.count(collection_name="primelands")

print(f"📊 Total vectors indexed: {count_result.count}")

✅ Embeddings initialized: text-embedding-3-large
🌐 Provider: openai
📊 Total chunks to embed: 691

🚀 Creating Qdrant vector store...

✅ Vector store created!
📊 Total vectors indexed: 691


## Index Sanity Check

In [19]:
# Cell 12: Index Sanity Check
print("🧪 Index Sanity Check\n")

# Verify count
count = count_result.count
print(f"✅ Collection contains {count} vectors")
assert count > 0, "❌ Collection is empty!"

# Test query
test_query = "kiribathgoda apartments"
print(f"\n🔍 Test query: '{test_query}'\n")

results = vectorstore.similarity_search_with_score(
    query=test_query,
    k=3
)

print("Top 3 results:")
for i, (doc, score) in enumerate(results, 1):
    print(f"\n{i}. Score: {score:.4f}")
    print(f"   URL: {doc.metadata['url']}")
    print(f"   Strategy: {doc.metadata['strategy']}")
    print(f"   Preview: {doc.page_content}...")

print("\n✅ Index sanity check passed!")
print(f"✅ Vector store persisted at: {VECTOR_DIR}")

🧪 Index Sanity Check

✅ Collection contains 691 vectors

🔍 Test query: 'kiribathgoda apartments'

Top 3 results:

1. Score: 0.4671
   URL: https://www.primelands.lk/apartment/YOLO-KIRIBATHGODA/en
   Strategy: late_chunk_base
   Preview: [![primelogo.png](https://www.primelands.lk/public/assets/images/primelogo.png)](https://www.primelands.lk/en)

[Home](https://www.primelands.lk/en)

[Lands](https://www.primelands.lk/land/en)

[Houses](https://www.primelands.lk/house/en)

[Apartments](https://www.primelands.lk/apartment/en)

[Portfolio Properties](https://www.primelands.lk/portfolio-property/en)

[Contact Us](https://www.primelands.lk/contact-us/en)

[Prime Residencies](https://www.primeresidencies.lk/)

[සිං   |](https://www.primelands.lk/apartment/YOLO-KIRIBATHGODA/sin)
[தமி](https://www.primelands.lk/apartment/YOLO-KIRIBATHGODA/tam)...

2. Score: 0.4670
   URL: https://www.primelands.lk/apartment/YOLO-KIRIBATHGODA/en
   Strategy: fixed
   Preview: [![primelogo.png](https://www.prime